# AQG trên GPU A100 80GB (Colab) — tự host LLM bằng vLLM

Notebook này thay chat2api/OpenAI bằng **hai model mở chạy trên chính GPU của Colab**:

| Vai trò | Model mặc định (preset `quality`) | Cổng |
|---|---|---|
| Writer, Distractor, Critic (đọc ảnh trang PDF) | `Qwen/Qwen3-VL-32B-Instruct-FP8` | 8000 |
| Solver độc lập **khác họ** (chỉ đọc đề bài) | `microsoft/phi-4` (FP8) | 8001 |

Hội đồng solver độc lập gồm cả phi-4 (khác họ) lẫn Qwen (cùng họ với generator), luật đồng thuận `all`:
chỉ cấp nhãn *independently_verified* khi **mọi** solver cùng ra giá trị của key; chỉ cần **một** solver
ra giá trị khác là câu bị chuyển người duyệt. Đây là phần trả lời cho câu hỏi của reviewer jPEd về lỗi tương quan.

**Các bước**

1. Kiểm GPU → cấu hình → clone repo → cài vLLM
2. Ghim snapshot trọng số (commit SHA trên Hugging Face) → bật 2 server vLLM
3. Kiểm tra nhanh (test, solver, ngân sách token ảnh)
4. **Thí nghiệm 1** — replay hội đồng solver trên 197 câu đã audit (vài phút, không sinh câu mới)
5. **Thí nghiệm 2** — sinh ~100 câu từ 6 tài liệu Toán 12
6. Hậu xử lý: tổng hợp, lỗi soạn đề (luật), phiếu audit mù, phiếu chấm giáo viên → lưu Drive
7. (Tuỳ chọn) mở API ra ngoài cho web app chạy ở máy bạn

Runtime: **Runtime → Change runtime type → A100 GPU** (cần gói Colab Pro/Pro+ hoặc pay-as-you-go, bật *High-RAM*).
Mọi kết quả ghi thẳng vào Google Drive nên mất kết nối thì chạy lại từ bước 1: phần đã xong được bỏ qua.

In [ ]:
#@title 1. Kiểm tra GPU
import subprocess
info = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
                       '--format=csv,noheader,nounits'], capture_output=True, text=True).stdout.strip()
print(info)
name, mem, driver = [x.strip() for x in info.split(',')]
assert 'A100' in name and int(mem) >= 79000, (
    f'Notebook này được tính ngân sách bộ nhớ cho A100 80GB, runtime hiện tại: {name} {mem} MiB')

In [ ]:
#@title 2. Cấu hình
import os, time, json
PRESET = 'quality'  #@param ['quality', 'awq_fallback', 'fast']
REPO = 'https://github.com/trantrien1/AQG.git'  #@param {type:'string'}
BRANCH = 'colab-gpu-reviewer-fixes'  #@param {type:'string'}
RUN_NAME = 'colab-' + time.strftime('%Y%m%d')  #@param {type:'raw'}
QUESTIONS_PER_DOC = 17  #@param {type:'integer'}
SEEDS = '42'  #@param {type:'string'}
USE_DRIVE = True  #@param {type:'boolean'}
DRIVE_DIR = '/content/drive/MyDrive/AQG_runs'  #@param {type:'string'}

# 6 tài liệu Toán 12 (Run C của paper) × 17 câu ≈ 102 câu.
DOCS = [
    'pdftest/corpus2/t12-daoham-p1.pdf',
    'pdftest/corpus2/t12-daoham-p2.pdf',
    'pdftest/corpus2/t12-khoi-dadien.pdf',
    'pdftest/corpus2/t12-mulog-tichphan.pdf',
    'pdftest/corpus2/t12-oxyz-p1.pdf',
    'pdftest/corpus2/t12-oxyz-p2.pdf',
]

# Ngân sách bộ nhớ (A100 80GB). Solver bật TRƯỚC: phi-4 FP8 nạp trọng số bf16
# rồi mới lượng tử hoá nên cần GPU còn trống lúc nạp.
PRESETS = {
    # Chất lượng cao nhất. FP8 trên A100 chạy qua kernel Marlin (W8A16).
    'quality': dict(
        gen='Qwen/Qwen3-VL-32B-Instruct-FP8', gen_util=0.66, gen_len=40960, gen_extra=[],
        solver='microsoft/phi-4', solver_util=0.24, solver_len=4096,
        solver_extra=['--quantization', 'fp8', '--enforce-eager'],
        dpi=96, max_pages=24, parallel=4),
    # Dự phòng khi server FP8 không lên được trên A100.
    'awq_fallback': dict(
        gen='Qwen/Qwen2.5-VL-32B-Instruct-AWQ', gen_util=0.62, gen_len=32768, gen_extra=[],
        solver='microsoft/phi-4', solver_util=0.24, solver_len=4096,
        solver_extra=['--quantization', 'fp8', '--enforce-eager'],
        dpi=96, max_pages=16, parallel=4),
    # Nhanh: generator 8B bf16, solver phi-4 bf16 (không lượng tử hoá).
    'fast': dict(
        gen='Qwen/Qwen3-VL-8B-Instruct', gen_util=0.46, gen_len=40960, gen_extra=[],
        solver='microsoft/phi-4', solver_util=0.44, solver_len=4096,
        solver_extra=['--enforce-eager'],
        dpi=110, max_pages=30, parallel=6),
}
P = PRESETS[PRESET]
GEN_PORT, SOLVER_PORT = 8000, 8001
print(json.dumps(P, indent=1))

In [ ]:
#@title 3. Google Drive (lưu kết quả)
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = f'{DRIVE_DIR}/{RUN_NAME}'
else:
    OUT = f'/content/runs/{RUN_NAME}'
os.makedirs(OUT, exist_ok=True)
print('Kết quả ghi vào:', OUT)

In [ ]:
#@title 4. Clone repo + cài vLLM (~5 phút)
%cd /content
!rm -rf /content/AQG && git clone -q --depth 1 -b "$BRANCH" "$REPO" /content/AQG
%cd /content/AQG/API
!git log --oneline -1
!pip -q install -U vllm
!pip -q install PyMuPDF networkx openpyxl python-docx pytest
!python -c "import vllm, torch, transformers, openai; print('vllm', vllm.__version__, '| torch', torch.__version__, '| transformers', transformers.__version__, '| openai', openai.__version__)"
import sys
sys.path.insert(0, '/content/AQG/API')

In [ ]:
#@title 5. Ghim snapshot trọng số + đặt biến môi trường cho pipeline
import secrets
from huggingface_hub import HfApi

try:  # tuỳ chọn: token HF trong Colab Secrets giúp tải nhanh/ít bị giới hạn hơn
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

api = HfApi()
REVISIONS = {m: api.model_info(m).sha for m in (P['gen'], P['solver'])}
print('Snapshot ghim:', json.dumps(REVISIONS, indent=1))

VLLM_KEY = os.environ.get('VLLM_KEY') or secrets.token_urlsafe(24)
GEN_URL = f'http://127.0.0.1:{GEN_PORT}/v1'
SOLVER_URL = f'http://127.0.0.1:{SOLVER_PORT}/v1'
PIPELINE_ENV = {
    'VLLM_KEY': VLLM_KEY,
    'AQG_LLM_PROVIDER': 'openai_compatible',
    'OPENAI_COMPATIBLE_BASE_URL': GEN_URL,
    'OPENAI_API_KEY': VLLM_KEY,
    'OPENAI_GENERATOR_MODEL': P['gen'],
    'OPENAI_JUDGE_MODEL': P['gen'],
    # Hội đồng: phi-4 (khác họ) + chính model sinh (cùng họ), luật 'all'.
    'AQG_INDEPENDENT_SOLVERS': f"{P['solver']}@{SOLVER_URL},{P['gen']}@{GEN_URL}",
    'AQG_INDEPENDENT_API_KEY': VLLM_KEY,
    'AQG_INDEPENDENT_CONSENSUS': 'all',
    'AQG_MODEL_REVISIONS': json.dumps(REVISIONS),
    # Tài liệu gửi dạng ảnh từng trang, đứng TRƯỚC prompt để vLLM dùng lại KV cache.
    'AQG_PDF_ATTACH_MODE': 'image',
    'AQG_PDF_IMAGE_DPI': str(P['dpi']),
    'AQG_PDF_IMAGE_MAX_PAGES': str(P['max_pages']),
    'AQG_ATTACHMENTS_FIRST': '1',
    'AQG_DIRECT_PDF_PARALLEL_SLOTS': str(P['parallel']),
    'AQG_LLM_TIMEOUT_SECONDS': '900',
    'AQG_LLM_RETRIES': '2',
    'PYTHONIOENCODING': 'utf-8',
}
os.environ.update(PIPELINE_ENV)
json.dump({k: v for k, v in PIPELINE_ENV.items() if 'KEY' not in k},
          open(f'{OUT}/pipeline_env.json', 'w'), indent=1)

In [ ]:
#@title 6. Hàm bật/tắt server vLLM
import subprocess, urllib.request
LOG_DIR = '/content/vllm_logs'
os.makedirs(LOG_DIR, exist_ok=True)
SERVERS = globals().get('SERVERS', {})

def start_vllm(name, model, port, util, max_len, extra, vision):
    if name in SERVERS and SERVERS[name].poll() is None:
        print(f'{name} đang chạy (pid {SERVERS[name].pid})'); return
    cmd = ['vllm', 'serve', model,
           '--revision', REVISIONS[model], '--tokenizer-revision', REVISIONS[model],
           '--host', '127.0.0.1', '--port', str(port), '--api-key', VLLM_KEY,
           '--gpu-memory-utilization', str(util), '--max-model-len', str(max_len),
           '--max-num-seqs', '8', '--enable-prefix-caching'] + list(extra)
    if vision:
        cmd += ['--limit-mm-per-prompt', json.dumps({'image': P['max_pages'] + 2, 'video': 0})]
    log = open(f'{LOG_DIR}/{name}.log', 'w')
    SERVERS[name] = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT,
                                     start_new_session=True)
    print(f'bật {name}: {model} (pid {SERVERS[name].pid}), log {LOG_DIR}/{name}.log')

def wait_ready(name, port, timeout=2400):
    t0 = time.time()
    while time.time() - t0 < timeout:
        if SERVERS[name].poll() is not None:
            print(open(f'{LOG_DIR}/{name}.log').read()[-4000:])
            raise RuntimeError(f'{name} đã thoát (mã {SERVERS[name].returncode}) — xem log ở trên')
        try:
            with urllib.request.urlopen(f'http://127.0.0.1:{port}/health', timeout=5) as r:
                if r.status == 200:
                    print(f'{name} sẵn sàng sau {time.time() - t0:.0f}s'); return
        except Exception:
            pass
        time.sleep(15)
    raise TimeoutError(f'{name} chưa sẵn sàng sau {timeout}s')

def stop_all():
    for name, proc in SERVERS.items():
        if proc.poll() is None:
            os.killpg(proc.pid, 15); print('dừng', name)

In [ ]:
#@title 7. Bật server solver (phi-4) rồi server generator (Qwen-VL) — 10–25 phút lần đầu
start_vllm('solver', P['solver'], SOLVER_PORT, P['solver_util'], P['solver_len'],
           P['solver_extra'], vision=False)
wait_ready('solver', SOLVER_PORT)
start_vllm('gen', P['gen'], GEN_PORT, P['gen_util'], P['gen_len'], P['gen_extra'], vision=True)
wait_ready('gen', GEN_PORT)
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

Nếu server **gen** thoát với lỗi liên quan tới FP8/Marlin hoặc *out of memory*: chạy `stop_all()`, đổi `PRESET` ở ô 2
thành `awq_fallback` (hoặc `fast`), rồi chạy lại ô 2 → 5 → 7. Nếu **solver** lỗi khi lượng tử hoá FP8, dùng preset `fast`.

In [ ]:
#@title 8. Kiểm tra nhanh: test đơn vị, solver, ngân sách token ảnh
!python -m pytest -q tests/test_independent_panel.py tests/test_panel_eval.py tests/test_verification_status.py tests/test_independent_target.py

from openai import OpenAI
from pipeline.independent_target import build_independent_target
from pipeline.llm_client import call_llm

for model, url in ((P['solver'], SOLVER_URL), (P['gen'], GEN_URL)):
    t = build_independent_target(
        'Tính tích phân của hàm số f(x) = 3x^2 + 2x trên đoạn [0; 2].',
        call_fn=lambda s, u, m=model, b=url: call_llm(s, u, model=m, temperature=0,
                                                      max_tokens=600, base_url=b, api_key=VLLM_KEY),
        model=model)
    print(f'{model}: definite={t.definite} value={t.value} expr={t.expression!r}  (đúng: 12)')

# Tài liệu dài nhất phải lọt ngân sách ngữ cảnh của server generator.
import fitz
from pipeline.direct_pdf.attach import build_pdf_image_content
longest = max(DOCS, key=lambda d: len(fitz.open(d)))
parts = build_pdf_image_content(open(longest, 'rb').read(), longest, 'Tài liệu có bao nhiêu trang?',
                                dpi=P['dpi'], max_pages=P['max_pages'], images_first=True)
client = OpenAI(base_url=GEN_URL, api_key=VLLM_KEY, timeout=900)
t0 = time.time()
resp = client.chat.completions.create(model=P['gen'], max_tokens=32, temperature=0,
                                      messages=[{'role': 'user', 'content': parts}])
used = resp.usage.prompt_tokens
print(f'{longest}: {len(parts) - 1} trang ảnh = {used} token prompt, {time.time() - t0:.0f}s')
print('trả lời:', resp.choices[0].message.content)
headroom = P['gen_len'] - used
print(f'còn {headroom} token cho prompt chữ + đầu ra (cần > ~8000)')
assert headroom > 8000, 'Hạ dpi hoặc max_pages trong PRESETS rồi chạy lại ô 2 và 5'

## Thí nghiệm 1 — replay hội đồng solver trên 197 câu đã audit

Không sinh câu mới. Mỗi solver giải lại đề của 197 câu đã có nhãn CAS (16 key sai); kết quả gpt-4o-mini đã lưu
trong artifact được đưa vào như solver `recorded`. Bảng kết quả cho biết: solver nào tái tạo key sai (lỗi tương quan),
κ giữa các cặp, và với mỗi hội đồng con: bao nhiêu key sai vẫn được cấp nhãn, bao nhiêu key đúng giữ được nhãn.

In [ ]:
#@title 9. Replay (vài phút)
REPLAY_OUT = f'{OUT}/replay'
!python scripts/replay_independent.py \
    --solver "{P['solver']}@{SOLVER_URL}" --solver "{P['gen']}@{GEN_URL}" \
    --api-key "$VLLM_KEY" --out "{REPLAY_OUT}" --workers 8
from IPython.display import Markdown, display
display(Markdown(open(f'{REPLAY_OUT}/replay_summary.md', encoding='utf-8').read()))

## Thí nghiệm 2 — sinh ~100 câu

Chạy nền; ô 11 theo dõi tiến độ (dừng ô 11 **không** dừng lượt sinh). Mỗi tài liệu ghi ra một file JSON khi xong,
nên nếu Colab ngắt kết nối: chạy lại ô 1 → 8 rồi ô 10 — tài liệu đã xong được bỏ qua.

In [ ]:
#@title 10. Bắt đầu sinh (chạy nền)
GEN_OUT = f'{OUT}/gen'
os.makedirs(GEN_OUT, exist_ok=True)
seeds = SEEDS.split()
cmd = [sys.executable, 'scripts/bench_suite.py', '--pdfs', *DOCS, '--arms', 'full_system',
       '--seeds', *seeds, '--n', str(QUESTIONS_PER_DOC), '--out', GEN_OUT]
GEN_LOG = f'{GEN_OUT}/suite_driver.log'
GEN_PROC = subprocess.Popen(cmd, stdout=open(GEN_LOG, 'a'), stderr=subprocess.STDOUT,
                            cwd='/content/AQG/API', start_new_session=True)
print('pid', GEN_PROC.pid, '| mục tiêu', len(DOCS) * len(seeds) * QUESTIONS_PER_DOC, 'câu')

In [ ]:
#@title 11. Theo dõi tiến độ (cập nhật mỗi phút)
import glob
from IPython.display import clear_output

def progress():
    done = 0
    for f in glob.glob(f'{GEN_OUT}/full_system/seed*/*.json'):
        if f.endswith('.manifest.json'):
            continue
        try:
            done += len(json.load(open(f, encoding='utf-8')).get('questions') or [])
        except Exception:
            pass
    running = ''
    logs = sorted(glob.glob(f'{GEN_OUT}/full_system/seed*/*.log'), key=os.path.getmtime)
    if logs:
        lines = [l for l in open(logs[-1], encoding='utf-8', errors='replace').read().splitlines()
                 if l.startswith('[cell]')]
        running = f'{os.path.basename(logs[-1])}: {lines[-1] if lines else "đang khởi động"}'
    return done, running

t0 = time.time()
while GEN_PROC.poll() is None:
    done, running = progress()
    clear_output(wait=True)
    print(f'{time.strftime("%H:%M:%S")} | đã giao {done} câu (tài liệu đã xong) | {time.time() - t0:.0f}s')
    print(running)
    print(''.join(open(GEN_LOG, encoding='utf-8').readlines()[-4:]))
    time.sleep(60)
print('XONG, mã thoát', GEN_PROC.returncode)
print(open(GEN_LOG, encoding='utf-8').read()[-3000:])

In [ ]:
#@title 12. Hậu xử lý: tổng hợp + lỗi soạn đề + phiếu audit mù + phiếu giáo viên
!python scripts/bench_suite.py --out "{GEN_OUT}" --aggregate-only
!python scripts/qeval.py --layer rules --corpus "{GEN_OUT}" --out "{OUT}/quality" | tail -30
!python scripts/qeval.py --layer position --corpus "{GEN_OUT}" --out "{OUT}/quality" | tail -15
!python scripts/audit_sample.py sample --run "{GEN_OUT}" --out "{OUT}/audit" --per-stratum 10
!python scripts/human_eval_packet.py export --run "{GEN_OUT}" --out "{OUT}/human_eval" --n 60 --reviewers gv1 gv2 gv3
display(Markdown(open(f'{GEN_OUT}/suite_summary.md', encoding='utf-8').read()))

In [ ]:
#@title 13. Hội đồng trên câu mới: từng solver một mình sẽ cấp nhãn khác đi thế nào?
import collections, itertools
from pipeline.independent_target import aggregate_panel
from pipeline.panel_eval import option_values, target_from_dict
from pipeline.verification_status import adjudicate

records = []
for f in sorted(glob.glob(f'{GEN_OUT}/full_system/seed*/*.json')):
    if not f.endswith('.manifest.json'):
        records += json.load(open(f, encoding='utf-8')).get('questions') or []
print(len(records), 'câu')

members = [m for m in PIPELINE_ENV['AQG_INDEPENDENT_SOLVERS'].split(',')]
names = [m.split('@')[0] for m in members]
table = collections.defaultdict(collections.Counter)
for rec in records:
    v = rec.get('verification') or {}
    panel = {m['model']: target_from_dict(m) for m in (v.get('independent') or {}).get('panel') or []}
    if len(panel) != len(names):
        table['(không đủ hội đồng)'][v.get('status')] += 1
        continue
    key, others = option_values(rec)
    for size in range(1, len(names) + 1):
        for subset in itertools.combinations(names, size):
            agg = aggregate_panel([target_from_dict(panel[n].to_dict()) for n in subset], 'all')
            adj = adjudicate(writer_verified=v.get('verified'), writer_engine=v.get('engine') or 'none',
                             independent=agg, keyed_value=key, distractor_values=others)
            table[' + '.join(s.split('/')[-1] for s in subset)][adj.status] += 1
statuses = ['independently_verified', 'consistency_confirmed', 'mismatch', 'refuted', 'non_verifiable']
print(f"{'cấu hình':45s}" + ''.join(f'{s[:14]:>16s}' for s in statuses))
for cfg_name, counts in table.items():
    print(f'{cfg_name:45s}' + ''.join(f'{counts.get(s, 0):16d}' for s in statuses))

### Việc cần làm sau khi có kết quả

* **Audit mù**: mở `audit/audit_sheet.csv` bằng Excel, điền cột `key_correct` (1 = key đúng, 0 = key sai, để trống nếu đề
  lỗi không có đáp án duy nhất). Tốt nhất hai người điền độc lập vào hai bản sao. Sau đó:
  `python scripts/audit_sample.py estimate --plan audit/audit_plan_SECRET.json --sheet a.csv b.csv`
  → tỉ lệ key sai từng tầng (Wilson CI), ước lượng phân tầng, κ giữa hai người, và `ground_truth_cas.json`.
* **Giáo viên**: gửi `human_eval/ratings_gvX.csv` + `HUONG_DAN_CHAM.txt` (KHÔNG gửi `_assignment_SECRET.json`).
  Nhận về → `import-csv` từng file → `aggregate`.
* Có nhãn audit rồi thì chạy lại tổng hợp có correctness:
  `python scripts/bench_suite.py --out gen --aggregate-only --ground-truth audit/ground_truth_cas.json`

In [ ]:
#@title 14. Nén kết quả (và tải về nếu không dùng Drive)
import shutil
archive = shutil.make_archive(f'/content/{RUN_NAME}', 'zip', OUT)
shutil.copytree(LOG_DIR, f'{OUT}/vllm_logs', dirs_exist_ok=True)
print(archive)
if not USE_DRIVE:
    from google.colab import files
    files.download(archive)

## (Tuỳ chọn) Dùng GPU này cho web app chạy ở máy bạn

Ô dưới mở hai đường hầm Cloudflare tạm thời tới hai server vLLM và in ra các dòng cần dán vào `API/.env` ở máy bạn.
API được bảo vệ bằng `VLLM_KEY`; đường hầm tắt khi runtime tắt. Lưu ý: điều khoản Colab hạn chế việc dùng runtime làm
dịch vụ web lâu dài — chỉ dùng cho thử nghiệm ngắn, ưu tiên chạy benchmark ngay trong notebook như trên.

In [ ]:
#@title 15. Mở đường hầm cho web app ở máy bạn
import re
if not os.path.exists('/content/cloudflared'):
    subprocess.run(['wget', '-q', '-O', '/content/cloudflared',
                    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'],
                   check=True)
    os.chmod('/content/cloudflared', 0o755)
TUNNELS = {}
for name, port in (('gen', GEN_PORT), ('solver', SOLVER_PORT)):
    log = f'{LOG_DIR}/tunnel-{name}.log'
    subprocess.Popen(['/content/cloudflared', 'tunnel', '--no-autoupdate', '--url', f'http://127.0.0.1:{port}'],
                     stdout=open(log, 'w'), stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(60):
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open(log).read())
        if m:
            TUNNELS[name] = m.group(0); break
        time.sleep(2)
print('# Dán vào API/.env ở máy bạn (thay cho khối chat2api):')
print('AQG_LLM_PROVIDER=openai_compatible')
print(f"OPENAI_COMPATIBLE_BASE_URL={TUNNELS['gen']}/v1")
print(f'OPENAI_API_KEY={VLLM_KEY}')
print(f"OPENAI_GENERATOR_MODEL={P['gen']}")
print(f"OPENAI_JUDGE_MODEL={P['gen']}")
print(f"AQG_INDEPENDENT_SOLVERS={P['solver']}@{TUNNELS['solver']}/v1,{P['gen']}@{TUNNELS['gen']}/v1")
print(f'AQG_INDEPENDENT_API_KEY={VLLM_KEY}')
for k in ('AQG_PDF_ATTACH_MODE', 'AQG_PDF_IMAGE_DPI', 'AQG_PDF_IMAGE_MAX_PAGES',
          'AQG_ATTACHMENTS_FIRST', 'AQG_LLM_TIMEOUT_SECONDS', 'AQG_MODEL_REVISIONS'):
    print(f'{k}={PIPELINE_ENV[k]}')

## Xử lý sự cố

| Triệu chứng | Cách xử lý |
|---|---|
| Server thoát, log có `CUDA out of memory` | `stop_all()`, hạ `gen_util`/`gen_len` hoặc đổi preset, chạy lại ô 2→5→7 |
| Server gen thoát với lỗi FP8/Marlin | preset `awq_fallback` |
| Log gen báo `max seq len ... larger than the maximum number of tokens that can be stored in KV cache` | trong preset: `gen_len=32768`, `max_pages=20`; chạy lại ô 2, 5, 7 |
| `maximum context length` / `At most N image(s)` trong log sinh | hạ `dpi` hoặc `max_pages` trong preset (ô 2), chạy lại ô 2, 5, 7 |
| Lượt sinh dừng với `401` | `VLLM_KEY` đổi sau khi server đã bật: `stop_all()` rồi bật lại ô 7 |
| Chạy chậm | xem `gen` log: nếu hàng đợi dài, hạ `parallel`; nếu GPU rảnh, tăng `parallel` |
| Mất kết nối Colab | chạy lại ô 1→8 rồi ô 10; tài liệu đã xong được bỏ qua |
| Kiểm tra lại cảnh báo độc lập | `gen/full_system/seed*/<doc>.manifest.json` → `notes.validity_warnings` |

In [ ]:
#@title 16. Tắt server khi xong (giải phóng GPU)
stop_all()